In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
import numpy as np
import io
import os
import sys
from PIL import Image
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from microfilm.microplot import microshow
from skimage.filters import gaussian, sobel
from skimage import measure, filters
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy import ndimage as ndi
from skimage.measure import label
from skimage.morphology import binary_opening
from skimage.segmentation import watershed

In [2]:
# with this peice of code, it will recognize the custom modules
project_root = "/Users/cgeyskens/Documents/code/phd/image-analysis/synapse-counting"
sys.path.append(project_root)

# custom modules
from synapse_counting import metadata, preprocessing, calc_synaptic_metrics

In [ ]:
# standard filters
file = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA1_SLM.czi"
pixel_size_um, _ , image_size_um = metadata.extract_metadata(file)
pre, post = preprocessing.extract_and_split(file, presynapse_channel = 0, postsynapse_channel = 1)
p = preprocessing.ImagePreprocessing(
            include_rolling_ball = True, radius = 10, # rolling ball parameters
            include_clahe = True, clip_limit = 0.005, kernel_size = 150, nbins = 265, # CLAHE parameters
            include_tophat = True, element_size = 5, # tophat parameters
            include_blur = True, sigma = 1, preserve_range = True # gaussian blur filters
            )
pre_1, post_1 = p.preprocess(pre, post)
presynapse_threshold = filters.threshold_otsu(pre_1)
presynapse_thresholded = pre_1 > presynapse_threshold

In [ ]:
def custom_watershed(binarized_image, sigma=3):
    """
    Wrapper for watershed segmentation, copied from Robert Haase notebook:
    https://haesleinhuepf.github.io/BioImageAnalysisNotebooks/20h_segmentation_post_processing/mimicking_imagej_watershed.html?highlight=watershed
    """
    distance = ndi.distance_transform_edt(binarized_image) # calculate distance image
    blurred_distance = gaussian(distance, sigma = sigma) # gaussian blur
    fp = np.ones((3,) * binarized_image.ndim) # neighbourhood size to find local maxima
    coords = peak_local_max(blurred_distance, footprint=fp, labels=binarized_image) # find local maxima
    mask = np.zeros(distance.shape, dtype=bool)
    mask[tuple(coords.T)] = True
    markers = label(mask)
    labels = watershed(-blurred_distance, markers, mask=binarized_image) # actual watershed
    edges_labels = sobel(labels)
    edges_binary = sobel(binarized_image)
    edges = np.logical_xor(edges_labels != 0, edges_binary != 0)
    almost = np.logical_not(edges) * binarized_image
    watershedded_image = binary_opening(almost)

    return watershedded_image

In [ ]:
def filter_out_small_puncta(binary_image, prop_df_with_area, labeled_image, puncta_size_threshold):
    prop_df_filtered = prop_df_with_area[prop_df_with_area['area'] > puncta_size_threshold]
    # create empty binary mask
    filtered_image = np.zeros_like(binary_image, dtype=bool)
    # iterate through the filtered properties
    for index, row in prop_df_filtered.iterrows():
        # get the label of the current region
        label = row['label']
        # set the pixels of the current region in the filtered_image to True
        filtered_image[labeled_image == label] = True

    return filtered_image

In [3]:
# input folder
input_folder = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images"

# get a list of files in that input_folder
file_list = os.listdir(input_folder)
print(file_list)

protein_and_synaptic_marker = "VCAM1_LacZ_VGLUT1_PSD95"

['CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA1_SLM.czi', 'CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA1_SO.czi', 'CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA1_SR.czi', 'CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA3_SL.czi', 'CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA3_SO.czi', 'CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA3_SR.czi', 'CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_DG_Hilus.czi', 'CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_DG_ML.czi', 'CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_VCAM1-gRNA_63X&3XzoomAiryscan_CA1_SLM.czi', 'CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_VCAM1-g

In [20]:
import json
with open("preprocess_params.json", "r") as file:
    # Read the file contents
    preprocess_params_dict = json.loads(file)



TypeError: the JSON object must be str, bytes or bytearray, not TextIOWrapper

In [18]:
params = preprocess_params_dict.get("VGLUT1_PSD95", {}).get("CA3_SO", {})
params

{'include_rolling_ball': 'True',
 'radius': 5,
 'include_blur': 'True',
 'sigma': 2,
 'include_clahe': 'True',
 'include_tophat': 'True',
 'element_size': 10,
 'watershed_sigma': 3,
 'puncta_size_threshold': 20}

In [17]:
params["radius"]

15

In [ ]:
# OPTIMIZATION OF PREPROCESSING FILTERS

# create synaptic_marker variable
synaptic_marker_string = protein_and_synaptic_marker.split("_")[-2:]
synaptic_marker = "_".join(synaptic_marker_string)

for filename in file_list:
    file_path = os.path.join(input_folder, filename)
    parts = filename.split('_')[-2:]
    result = "_".join(parts)
    print(result)
    
    if synaptic_marker == "VGLUT1_PSD95":
        if result in {"CA3_SL.czi", "DG_Hilus.czi"}:
            radius, element_size, blur_sigma, watershed_sigma, puncta_pix_size = 15, 20, 2, 3, 50
        elif result in {"CA1_SO.czi", "CA3_SO.czi"}:
            radius, element_size, blur_sigma, watershed_sigma, puncta_pix_size = 5, 10, 2, 3, 20
        elif result in {"CA1_SR.czi", "CA3_SR.czi"}:
            radius, element_size, blur_sigma, watershed_sigma, puncta_pix_size = 10, 10, 2, 3, 20
        elif result == "CA1_SLM.czi":
            radius, element_size, blur_sigma, watershed_sigma, puncta_pix_size = 5, 5, 2, 3, 20
        elif result == "DG_ML.czi":
            radius, element_size, blur_sigma, watershed_sigma, puncta_pix_size = 5, 15, 2, 3, 20
    elif synaptic_marker == "VGLUT2_PSD95":
        if result in {"Cortex_L4.czi"}:
            radius, element_size, blur_sigma, watershed_sigma, puncta_pix_size = 10, 10, 2, 3, 20
        elif result in {"CA2_SP.czi", "DG_GC.czi", "Subiculum_SP.czi"}:
            radius, element_size, blur_sigma, watershed_sigma, puncta_pix_size = 15, 10, 2, 3, 50

    
    print(f"Processing with radius={radius}, element_size={element_size}, blur_sigma={blur_sigma}, watershed_sigma={watershed_sigma}, puncta_pix_size={puncta_pix_size}")
    # extract metadata
    pixel_size_um, _ , image_size_um = metadata.extract_metadata(file_path)
    # splitting the channels
    pre, post = preprocessing.extract_and_split(file_path, presynapse_channel = 0, postsynapse_channel = 1)
    
    # preprocessing
    p = preprocessing.ImagePreprocessing(
        include_rolling_ball=True, radius=radius,
        include_blur=True, sigma = blur_sigma, preserve_range = True,
        include_clahe=True,
        include_tophat=True, element_size = element_size
        )
    pre_1, post_1 = p.preprocess(pre, post)
            
    # thresholding and watershed segmentation
    presynapse_threshold, postsynapse_threshold = preprocessing.thresholding(pre_1, post_1, threshold_algorithm="triangle")
    presynapse_watersheded = preprocessing.custom_watershed(presynapse_threshold, sigma = watershed_sigma)
    postsynapse_watersheded = preprocessing.custom_watershed(postsynapse_threshold, sigma = watershed_sigma)
    
    # getting the data
    presynapse_image_mfi, postsynapse_image_mfi = calc_synaptic_metrics.mfi_synapse(pre, post)
    puncta_results, pre_labeled , post_labeled , pre_prop, post_prop = calc_synaptic_metrics.puncta_metrics(presynapse_watersheded, postsynapse_watersheded, image_size_um, pixel_size_um)

    pre_filtered_image = filter_out_small_puncta(binary_image=presynapse_watersheded, 
                                                prop_df_with_area=pre_prop,
                                                labeled_image=pre_labeled,
                                                puncta_size_threshold = puncta_pix_size)
    post_filtered_image = filter_out_small_puncta(binary_image=postsynapse_watersheded, 
                                                prop_df_with_area=post_prop,
                                                labeled_image=post_labeled,
                                                puncta_size_threshold = puncta_pix_size)

    # Display the pre-processed and post-processed images
    fig, axes = plt.subplots(4, 2, figsize=(12, 16))

    axes[0, 0].imshow(pre, cmap='gray')
    axes[0, 0].set_title('Original Presynapse Image')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(post, cmap='gray')
    axes[0, 1].set_title('processed image')
    axes[0, 1].axis('off')

    axes[1, 0].imshow(pre_1, cmap='gray')
    axes[1, 0].set_title('Pre-processed Presynapse Image')
    axes[1, 0].axis('off')

    axes[1, 1].imshow(post_1, cmap='gray')
    axes[1, 1].set_title('Pre-processed Postsynapse Image')
    axes[1, 1].axis('off')

    axes[2, 0].imshow(presynapse_watersheded, cmap='gray')
    axes[2, 0].set_title('Binarized Presynapse Image')
    axes[2, 0].axis('off')

    axes[2, 1].imshow(postsynapse_watersheded, cmap='gray')
    axes[2, 1].set_title('Binarized Postsynapse Image')
    axes[2, 1].axis('off')

    axes[3, 0].imshow(pre_filtered_image, cmap='gray')
    axes[3, 0].set_title('pre_filtered_image')
    axes[3, 0].axis('off')

    axes[3, 1].imshow(post_filtered_image, cmap='gray')
    axes[3, 1].set_title('post_filtered_image')
    axes[3, 1].axis('off')

    plt.tight_layout()
    plt.show()

    

In [ ]:
import os
import matplotlib.pyplot as plt

# Define the parameter ranges
radius_range = range(10, 16, 5)
element_size_range = range(5, 16, 5)
blur_sigma_range = range(1, 3)
watershed_sigma_range = range(2, 5)

# Iterate over each file
for filename in file_list:
    file_path = os.path.join(input_folder, filename)
    parts = filename.split('_')[-2:]
    result = "_".join(parts)
    print(result)
    
    if result == "DG_GC.czi":
        # Iterate over all combinations of parameters
        for radius in radius_range:
            for element_size in element_size_range:
                for blur_sigma in blur_sigma_range:
                    for watershed in watershed_sigma_range:

                        print(f"Processing with radius={radius}, element_size={element_size}, blur_sigma={blur_sigma}, watershed_sigma={watershed_sigma}")

                        # extract metadata
                        pixel_size_um, _, image_size_um = metadata.extract_metadata(file_path)
                        # splitting the channels
                        pre, post = preprocessing.extract_and_split(file_path, presynapse_channel=0, postsynapse_channel=1)

                        # preprocessing
                        p = preprocessing.ImagePreprocessing(
                            include_rolling_ball=True, radius=radius,
                            include_blur=True, sigma=blur_sigma, preserve_range=True,
                            include_clahe=True,
                            include_tophat=True, element_size=element_size
                        )
                        pre_1, post_1 = p.preprocess(pre, post)

                        # thresholding and watershed segmentation
                        presynapse_threshold, postsynapse_threshold = preprocessing.thresholding(pre_1, post_1, threshold_algorithm="triangle")
                        presynapse_watersheded = preprocessing.custom_watershed(presynapse_threshold, sigma=watershed_sigma)
                        postsynapse_watersheded = preprocessing.custom_watershed(postsynapse_threshold, sigma=watershed_sigma)

                        # Display the pre-processed and post-processed images
                        fig, axes = plt.subplots(3, 2, figsize=(12, 16))

                        axes[0, 0].imshow(pre, cmap='gray')
                        axes[0, 0].set_title('Original Presynapse Image')
                        axes[0, 0].axis('off')

                        axes[0, 1].imshow(post, cmap='gray')
                        axes[0, 1].set_title('Original Postsynapse Image')
                        axes[0, 1].axis('off')

                        axes[1, 0].imshow(pre_1, cmap='gray')
                        axes[1, 0].set_title('Pre-processed Presynapse Image')
                        axes[1, 0].axis('off')

                        axes[1, 1].imshow(post_1, cmap='gray')
                        axes[1, 1].set_title('Pre-processed Postsynapse Image')
                        axes[1, 1].axis('off')

                        axes[2, 0].imshow(presynapse_watersheded, cmap='gray')
                        axes[2, 0].set_title('Binarized Presynapse Image')
                        axes[2, 0].axis('off')

                        axes[2, 1].imshow(postsynapse_watersheded, cmap='gray')
                        axes[2, 1].set_title('Binarized Postsynapse Image')
                        axes[2, 1].axis('off')

                        plt.tight_layout()
                        plt.show()
                else:
                    pass


In [21]:
import json

data = {
    "VGLUT1_PSD95": {
        "CA1_SO": {
            "include_rolling_ball": "True", "radius": 5,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 10,
            "watershed_sigma": 3,
            "puncta_size_threshold": 20, 
            "threshold_algorithm": "triangle"
        },
        "CA1_SR": {
            "include_rolling_ball": "True", "radius": 10,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 10,
            "watershed_sigma": 3,
            "puncta_size_threshold": 20,
            "threshold_algorithm": "triangle"
        },
        "CA1_SLM": {
            "include_rolling_ball": "True", "radius": 5,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 5,
            "watershed_sigma": 3,
            "puncta_size_threshold": 20,
            "threshold_algorithm": "triangle"
        },
        "CA3_SO": {
            "include_rolling_ball": "True", "radius": 5,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 10,
            "watershed_sigma": 3,
            "puncta_size_threshold": 20,
            "threshold_algorithm": "triangle"
        },
        "CA3_SL": {
            "include_rolling_ball": "True", "radius": 15,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 20,
            "watershed_sigma": 3,
            "puncta_size_threshold": 50,
            "threshold_algorithm": "triangle"
        },
        "CA3_SR": {
            "include_rolling_ball": "True", "radius": 10,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 10,
            "watershed_sigma": 3,
            "puncta_size_threshold": 20,
            "threshold_algorithm": "triangle"
        },
        "DG_Hilus": {
            "include_rolling_ball": "True", "radius": 15,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 20,
            "watershed_sigma": 3,
            "puncta_size_threshold": 50,
            "threshold_algorithm": "triangle"
        },
        "DG_ML": {
            "include_rolling_ball": "True", "radius": 5,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 15,
            "watershed_sigma": 3,
            "puncta_size_threshold": 20,
            "threshold_algorithm": "triangle"
        }
    },

    "VGLUT2_PSD95": {
        "Cortex_L4": {
            "include_rolling_ball": "True", "radius": 10,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 10,
            "watershed_sigma": 3,
            "puncta_size_threshold": 20,
            "threshold_algorithm": "triangle"
        },
        "CA2_SP": {
            "include_rolling_ball": "True", "radius": 15,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 10,
            "watershed_sigma": 3,
            "puncta_size_threshold": 50,
            "threshold_algorithm": "triangle"
        },
        "DG_GC": {
            "include_rolling_ball": "True", "radius": 15,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 10,
            "watershed_sigma": 3,
            "puncta_size_threshold": 50,
            "threshold_algorithm": "triangle"
        },
        "Subiculum_SP": {
            "include_rolling_ball": "True", "radius": 15,
            "include_blur": "True", "sigma": 2,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 10,
            "watershed_sigma": 3,
            "puncta_size_threshold": 50,
            "threshold_algorithm": "triangle"
        }
    }
}

def get_regions(data, key):
    if key in data:
        return list(data[key].keys())
    else:
        return []

# Get regions for VGLUT1_PSD95
vglut1_regions = get_regions(data, "VGLUT1_PSD95")
print("VGLUT1_PSD95 regions:", vglut1_regions)

# Get regions for VGLUT2_PSD95
vglut2_regions = get_regions(data, "VGLUT2_PSD95")
print("VGLUT2_PSD95 regions:", vglut2_regions)


VGLUT1_PSD95 regions: ['CA1_SO', 'CA1_SR', 'CA1_SLM', 'CA3_SO', 'CA3_SL', 'CA3_SR', 'DG_Hilus', 'DG_ML']
VGLUT2_PSD95 regions: ['Cortex_L4', 'CA2_SP', 'DG_GC', 'Subiculum_SP']
